# Boltzmann Learning — constrained-MaxEnt refit by moment matching

The production models are fit by **pseudo-likelihood** (`fit_pl_ising`), which matches each
spin's *conditional* but not the model's *moments*. The `pl_three_body_check` diagnostic exposed
the consequence: the PL species model's marginals over-concentrate (top model marginal ≈ 0.70
vs empirical ≈ 0.44), so a Schneidman-style three-body test is meaningless on it — that test
*requires* a fit whose 1st + 2nd moments equal the data's by construction.

This notebook fits `(J, h)` by **Boltzmann learning** (`fit_boltzmann_ising`): stochastic
gradient ascent on the regularized log-likelihood, with the model moments estimated from a
persistent bank of swap-move MCMC chains (PCD) that samples the *constrained* team ensemble
(fixed size + species/item uniqueness). The bank is stepped fully vectorized across chains
(batched NumPy) and moments are accumulated sparsely over each team's on-bits, so the loop is
fast even at species+item scale.

It then delivers the payoff:
1. **convergence** — moment residual falls toward the sampling-noise floor;
2. **moment reconstruction** — how much better the Boltzmann fit reproduces `⟨sᵢ⟩` / `⟨sᵢsⱼ⟩`
   than PL (quantifying the PL approximation gap);
3. **(J, h) comparison** — where the couplings/fields moved;
4. **the now-valid three-body test** — with moments matched, the 3-point scatter is a real
   test of pairwise sufficiency.

All knobs pull from `k2dex.constants`, so this tracks the production weighting / λ.

> **On the gauge:** on the fixed-size manifold `(J, h)` is gauge-degenerate
> (`h → h + c·1`, `J_ij → J_ij + a_i + a_j`). The L2/L1 regularizer (toward zero) is not
> gauge-invariant, so it selects the minimum-norm representative and pins the gauge. Keep
> `reg_lambda > 0` for a unique, comparable fit. (Set it small if you want the tightest
> possible moment match — larger reg deliberately shrinks `(J, h)`, trading moment fidelity
> for stability.)

In [ ]:
from __future__ import annotations

from itertools import combinations

import numpy as np
import matplotlib.pyplot as plt

from k2dex import tournament_ingest
from k2dex.constants import (
    TEAM_SIZE, RECENCY_TAU_DAYS, IN_PERSON_WEIGHT,
    SPECIES_LR_LAMBDA, SPECIES_ITEM_LR_LAMBDA,
)
from k2dex.loaders import (
    team_weights, format_pair, build_species_model, build_species_item_model,
)
from k2dex.models import fit_boltzmann_ising, empirical_moments
from k2dex.sampling import parallel_tempered_mcmc, estimate_moments

# Reg M-A: the mature, data-rich format this diagnostic needs (M-B is still
# sparse / warm-started). All weighting + λ come from k2dex.constants.
REG = "M-A"

## Inputs

`build_inputs` reuses the production loaders to get the PL fit `(J_pl, h_pl)`, vocab, and the
uniqueness lookups, then reconstructs the design matrix `X` and weights `w` on that same vocab
to compute the empirical moments (`m_data`, `C_data`) the Boltzmann fit targets. For the
**species** model the ensemble needs no uniqueness lookups (species are distinct per vocab
entry); for **species+item** the no-duplicate-species / no-duplicate-item constraints are
passed through.

In [ ]:
def build_inputs(model_type: str) -> dict:
    builder = build_species_model if model_type == "species" else build_species_item_model
    vocab, m_loader, J_pl, h_pl, _tc, species_of, item_of, latest = builder(regulation=REG)
    uniq = (None, None) if model_type == "species" else (species_of, item_of)

    tours = tournament_ingest.load_cached_tournaments(regulation=REG)
    obs = tournament_ingest.all_team_observations(tours)
    w = team_weights(obs, reference_date=latest,
                     recency_tau=RECENCY_TAU_DAYS, in_person_multiplier=IN_PERSON_WEIGHT)

    V = len(vocab)
    X = np.zeros((len(obs), V), dtype=np.int8)
    if model_type == "species":
        idx = {name: i for i, name in enumerate(vocab)}
        teams = tournament_ingest.species_only_teams([o.members for o in obs])
        for ti, team in enumerate(teams):
            for name in team:
                j = idx.get(name)
                if j is not None:
                    X[ti, j] = 1
    else:
        idx = {(s, it): i for i, (s, it) in enumerate(zip(species_of, item_of))}
        for ti, o in enumerate(obs):
            for pair in o.members:
                j = idx.get(pair)
                if j is not None:
                    X[ti, j] = 1

    m_data, C_data = empirical_moments(X, w)
    # sanity: reconstructed marginals match the loader's weighted m
    assert np.abs(m_data - m_loader).max() < 1e-9, "vocab/X reconstruction drifted"
    return dict(model_type=model_type, vocab=vocab, V=V, X=X, w=w,
                J_pl=J_pl, h_pl=h_pl, species_of=species_of, item_of=item_of,
                uniq=uniq, m_data=m_data, C_data=C_data)

## Helpers — model-moment estimation and the diagnostic panels

`model_moments` draws the model's `⟨sᵢ⟩` / `⟨sᵢsⱼ⟩` at **T = 1** (the cold chain *must* be at
T = 1 to sample `P ∝ exp(-H)`; hotter rungs only aid mixing). `recon_panel` scatters a fit's
reconstructed moments against the data, side by side for the PL and Boltzmann fits.

In [ ]:
def model_moments(J, h, uniq, *, n_runs=8, n_steps=60_000, burn_in=10_000, thin=25, seed=1):
    res = estimate_moments(J, h, TEAM_SIZE, species_of=uniq[0], item_of=uniq[1],
                           n_runs=n_runs, n_steps=n_steps, burn_in=burn_in, thin=thin, seed=seed)
    assert res is not None
    return res[0], res[1], res[2]


def _offdiag(C, V):
    iu = np.triu_indices(V, k=1)
    return C[iu]


def recon_panel(inp, fits, title):
    # fits: list of (label, m_model, C_model). Scatters each fit's marginals
    # and pair moments against the empirical data.
    V = inp["V"]
    m_d, C_d = inp["m_data"], inp["C_data"]
    cd = _offdiag(C_d, V)
    n = len(fits)
    fig, axes = plt.subplots(2, n, figsize=(5.2 * n, 9))
    for col, (label, m_m, C_m) in enumerate(fits):
        a = axes[0, col]
        a.scatter(m_d, m_m, s=10, alpha=0.6)
        lim = max(m_d.max(), m_m.max()) * 1.05
        a.plot([0, lim], [0, lim], "k--", lw=0.8)
        a.set_xlabel("empirical ⟨sᵢ⟩"); a.set_ylabel(f"{label} ⟨sᵢ⟩")
        a.set_title(f"{label} marginals  r={np.corrcoef(m_d, m_m)[0,1]:.4f}\n"
                    f"max ⟨sᵢ⟩: data {m_d.max():.3f} / model {m_m.max():.3f}")
        a = axes[1, col]
        cm = _offdiag(C_m, V)
        a.scatter(cd, cm, s=4, alpha=0.3)
        lo, hi = min(cd.min(), cm.min()), max(cd.max(), cm.max())
        a.plot([lo, hi], [lo, hi], "k--", lw=0.8)
        a.set_xlabel("empirical ⟨sᵢsⱼ⟩"); a.set_ylabel(f"{label} ⟨sᵢsⱼ⟩")
        a.set_title(f"{label} pair moments  r={np.corrcoef(cd, cm)[0,1]:.4f}")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

## Species model

Fit the Boltzmann `(J, h)`, warm-started from the PL fit. `n_sweeps` controls mixing: the
*mean* residual converges fast, but the *max* residual on the few highest-marginal features is
mixing-limited, so it wants a high sweep count (cheap under batched NumPy).

In [ ]:
sp = build_inputs("species")
print(f"species: V={sp['V']}, teams={sp['X'].shape[0]:,}")

J_bz, h_bz, hist = fit_boltzmann_ising(
    sp["X"], team_size=TEAM_SIZE, sample_weight=sp["w"],
    init_J=sp["J_pl"], init_h=sp["h_pl"],
    species_of=sp["uniq"][0], item_of=sp["uniq"][1],
    reg="l2", reg_lambda=1e-3,
    n_iters=500, lr=0.02, n_chains=400, n_sweeps=200, n_burn=400, seed=0,
)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist["mean_resid_m"], label="mean |Δ⟨sᵢ⟩|")
ax[0].plot(hist["max_resid_m"], label="max |Δ⟨sᵢ⟩|")
ax[0].set_title("marginal residual"); ax[0].set_xlabel("iteration"); ax[0].legend()
ax[1].plot(hist["mean_resid_C"], label="mean |Δ⟨sᵢsⱼ⟩|")
ax[1].plot(hist["max_resid_C"], label="max |Δ⟨sᵢsⱼ⟩|")
ax[1].set_title("pair-moment residual"); ax[1].set_xlabel("iteration"); ax[1].legend()
for a in ax:
    a.set_yscale("log")
fig.suptitle(f"convergence (local-acc {np.mean(hist['local_accept']):.2f}, replica-swap {np.mean(hist['swap_accept']):.2f})")
fig.tight_layout(); plt.show()

### Payoff 1 — moment reconstruction (PL vs Boltzmann)

The Boltzmann columns should land on the diagonal (it matched these moments by construction);
the PL columns reveal the gap — in particular the over-concentrated top marginal.

In [ ]:
m_pl, C_pl, d_pl = model_moments(sp["J_pl"], sp["h_pl"], sp["uniq"])
m_bz, C_bz, d_bz = model_moments(J_bz, h_bz, sp["uniq"])
print(f"PL sample swap-accept {d_pl['swap_accept']:.3f} | "
      f"Boltzmann sample swap-accept {d_bz['swap_accept']:.3f}")
recon_panel(sp, [("PL", m_pl, C_pl), ("Boltzmann", m_bz, C_bz)],
            "Species: moment reconstruction vs data")

### Payoff 2 — how far did (J, h) move?

In [ ]:
V = sp["V"]
iu = np.triu_indices(V, k=1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].scatter(sp["J_pl"][iu], J_bz[iu], s=3, alpha=0.3)
lo, hi = sp["J_pl"][iu].min(), sp["J_pl"][iu].max()
ax[0].plot([lo, hi], [lo, hi], "k--", lw=0.8)
ax[0].set_xlabel("J_pl"); ax[0].set_ylabel("J_boltzmann"); ax[0].set_title("couplings")
ax[1].scatter(sp["h_pl"], h_bz, s=10, alpha=0.5)
lo, hi = sp["h_pl"].min(), sp["h_pl"].max()
ax[1].plot([lo, hi], [lo, hi], "k--", lw=0.8)
ax[1].set_xlabel("h_pl"); ax[1].set_ylabel("h_boltzmann"); ax[1].set_title("fields")
fig.tight_layout(); plt.show()

# Biggest field movers.
dh = h_bz - sp["h_pl"]
order = np.argsort(-np.abs(dh))[:12]
print("largest field shifts (feature: h_pl -> h_bz):")
for i in order:
    print(f"  {sp['vocab'][i]:<24} {sp['h_pl'][i]:+.3f} -> {h_bz[i]:+.3f}  (Δ {dh[i]:+.3f})")

### Payoff 3 — the three-body test (now valid)

With 1st + 2nd moments matched, the connected 3-point correlations are a genuine test of
whether the pairwise model suffices. We compare data vs the Boltzmann model (and, for
contrast, vs PL — whose mismatch is partly mechanical because its lower moments are off).
Mirrors `pl_three_body_check.ipynb`, but with the cold chain correctly at **T = 1**.

In [ ]:
K_TOP = 50
N_RUNS, N_STEPS, BURN_IN, THIN = 20, 80_000, 10_000, 25


def sample_model(J, h, uniq, seed0=0):
    ladder = np.geomspace(1.0, 3.0, 8)   # cold chain at T=1 samples P ∝ exp(-H)
    rng = np.random.default_rng(seed0)
    pooled = []
    for _ in range(N_RUNS):
        res = parallel_tempered_mcmc(J, h, TEAM_SIZE, [], [], 1.0, ladder,
                                     N_STEPS, BURN_IN, 10, int(rng.integers(2**31)),
                                     species_of=uniq[0], item_of=uniq[1])
        assert res is not None
        pooled.append(res[0][::THIN])
    return np.concatenate(pooled).astype(np.float64)


def triple_product_mean(A, W):
    K = A.shape[1]
    out = np.empty((K, K, K))
    Aw = A * W[:, None]
    for i in range(K):
        out[i] = (A * Aw[:, [i]]).T @ A
    return out


def connected_3pt(X, w, idx):
    W = w / w.sum()
    Xs = X[:, idx].astype(np.float64)
    Xc = Xs - W @ Xs
    return triple_product_mean(Xc, W)


idx = np.argsort(sp["m_data"])[::-1][:K_TOP]
trips = np.array(list(combinations(range(K_TOP), 3)))
t_data = connected_3pt(sp["X"], sp["w"], idx)[trips[:, 0], trips[:, 1], trips[:, 2]]

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for col, (label, J, h) in enumerate([("PL", sp["J_pl"], sp["h_pl"]),
                                      ("Boltzmann", J_bz, h_bz)]):
    S = sample_model(J, h, sp["uniq"], seed0=col + 1)
    t_m = connected_3pt(S, np.ones(S.shape[0]), idx)[trips[:, 0], trips[:, 1], trips[:, 2]]
    a = ax[col]
    a.scatter(t_data, t_m, s=3, alpha=0.25)
    lo, hi = min(t_data.min(), t_m.min()), max(t_data.max(), t_m.max())
    a.plot([lo, hi], [lo, hi], "k--", lw=0.8)
    a.set_xlabel("data $T_{ijk}$"); a.set_ylabel(f"{label} $T_{{ijk}}$")
    a.set_title(f"{label}: connected 3-point  r={np.corrcoef(t_data, t_m)[0,1]:.3f}")
fig.suptitle(f"Species three-body test (top {K_TOP} features, {len(trips)} triplets)")
fig.tight_layout(); plt.show()

## Species + item

Same pipeline at pair scale. This vocab is much larger, so:
- pass the real uniqueness lookups (features share species and items);
- **support-gate** the couplings — only fit `J_ij` for pairs that co-occur often enough in the
  corpus (the rest stay at the PL warm-start), which removes pure-noise directions and cuts
  cost (this mirrors the v2 "support-gated ΔJ" idea).

Heavier than the species fit; tune `n_iters` / `n_sweeps` to taste.

In [ ]:
pr = build_inputs("species_item")
print(f"species+item: V={pr['V']}, teams={pr['X'].shape[0]:,}")

# Support mask: pairs co-occurring >= SUPPORT_MIN times in the corpus.
SUPPORT_MIN = 20
cooc = pr["X"].astype(np.int64).T @ pr["X"].astype(np.int64)
support_mask = cooc >= SUPPORT_MIN
print(f"fit {int(np.triu(support_mask, 1).sum()):,} of {pr['V']*(pr['V']-1)//2:,} "
      f"possible couplings (support >= {SUPPORT_MIN})")

J_bz_pr, h_bz_pr, hist_pr = fit_boltzmann_ising(
    pr["X"], team_size=TEAM_SIZE, sample_weight=pr["w"],
    init_J=pr["J_pl"], init_h=pr["h_pl"],
    species_of=pr["uniq"][0], item_of=pr["uniq"][1],
    support_mask=support_mask, reg="l2", reg_lambda=1e-3,
    n_iters=500, lr=0.02, n_chains=400, n_sweeps=150, n_burn=400, seed=0,
)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist_pr["mean_resid_m"], label="mean |Δ⟨sᵢ⟩|")
ax[0].plot(hist_pr["max_resid_m"], label="max |Δ⟨sᵢ⟩|")
ax[0].set_title("marginal residual"); ax[0].set_xlabel("iteration"); ax[0].legend()
ax[1].plot(hist_pr["mean_resid_C"], label="mean |Δ⟨sᵢsⱼ⟩|")
ax[1].plot(hist_pr["max_resid_C"], label="max |Δ⟨sᵢsⱼ⟩|")
ax[1].set_title("pair-moment residual"); ax[1].set_xlabel("iteration"); ax[1].legend()
for a in ax:
    a.set_yscale("log")
fig.suptitle(f"species+item convergence (local-acc {np.mean(hist_pr['local_accept']):.2f}, replica-swap {np.mean(hist_pr['swap_accept']):.2f})")
fig.tight_layout(); plt.show()

m_pl_pr, C_pl_pr, _ = model_moments(pr["J_pl"], pr["h_pl"], pr["uniq"],
                                    n_runs=6, n_steps=80_000, burn_in=15_000)
m_bz_pr, C_bz_pr, _ = model_moments(J_bz_pr, h_bz_pr, pr["uniq"],
                                    n_runs=6, n_steps=80_000, burn_in=15_000)
recon_panel(pr, [("PL", m_pl_pr, C_pl_pr), ("Boltzmann", m_bz_pr, C_bz_pr)],
            "Species+item: moment reconstruction vs data")

## Productionizing a Boltzmann fit (optional)

To load a Boltzmann model in the webapp picker for hands-on experimentation, build it through
the same precompute pipeline (no TS / parity changes — the artifact form is identical, only the
fitted numbers and `meta.fit.method` differ):

```bash
python scripts/precompute.py --display-name "Reg M-A Species Boltzmann" \
    --regulation M-A --type species --method boltzmann \
    --bz-iters 500 --bz-sweeps 200 --bz-reg-lambda 1e-3

python scripts/precompute.py --display-name "Reg M-A Species @ Item Boltzmann" \
    --regulation M-A --type species_item --method boltzmann \
    --bz-iters 500 --bz-sweeps 150 --bz-support-min-count 20

python scripts/precompute.py --generate-manifest
```

The Boltzmann hyperparameters are recorded in `meta.json:fit.boltzmann`, so `--recompute`
rebuilds the artifact from its own parameters like any other model.

> **Calibration note (if you expose one as a webapp model):** the app applies
> `field_weight · h`, and the field-weight defaults were tuned to PL's over-weighted `h`. A
> moment-matched model reproduces marginals at `field_weight ≈ 1`, so its preferred field
> weight will sit higher than the PL model's. Re-check `energy_discrimination.ipynb` before
> leaning on the old defaults.